### Import Dependencies

In [1]:
import os
from pathlib import Path

import instructor
from dotenv import load_dotenv
from google import genai
from pydantic import BaseModel, Field
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

# Find the project root and load its .env file for notebook kernels started in either folder.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook kernel inside the project directory.")
load_dotenv(PROJECT_ROOT / ".env", override=True)

google_api_key = (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY") or "").strip()
if not google_api_key:
    raise RuntimeError("Set GOOGLE_API_KEY or GEMINI_API_KEY in the project .env.")

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "Amazon-items-minilm-l6-v2"
gemini_client = genai.Client(api_key=google_api_key)
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

/Users/ganesh.m/Projects/ai-engineering-bootcamp/project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7661.30it/s]


### Mock Example

In [4]:
prompt = """
You are a helpful assistant.
Return an answer to the question.
Question: what is your name?"""

In [5]:
response = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents="What is your name?",
    config={"system_instruction": "You are a helpful assistant. Answer the user's question."},
)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


I don't have a name! I am a large language model trained by Google. You can just call me AI.


In [6]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="I don't have a name! I am a large language model trained by Google. You can just call me AI.",
            thought_signature=b'\x12`\n^\x01i\x14}\x135\xe0\x9dx\x0b\x051\x15\xb1g8\xa0\xb8\xcd\x03\xc8\xc6\x10\x99\xf9\xea\xded\xd6[am\xae/\x03\xaa\xc1\xc1\r\xd2\xb6\xea\xb5\xb8\x08n\x0e~da\xe1\xa3\xc2\x9f\xb0\xef\xc4\xe7\xcf\xe3Q\xf7\x90\xb2\xf54\xcf\xaf&\x18\xbf\xc9\xc4Y\x03\x02yw`\xe7%\xff\xe9\xae!\x19\xd0;\xccq*s!a'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-3.5-flash-lite',
  response_id='IRPmavfYLMDb4t4Ps4Xi6Qc',
  sdk_http_response=HttpResponse(
    headers=<dict len=12>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=25,
    prompt_token_count=20,
    prompt_tokens_detail

### Instructor with Gemini (Structured Outputs)

In [7]:
client = instructor.from_provider(
    f"google/{GEMINI_MODEL}",
    mode=instructor.Mode.TOOLS,
)

In [8]:
class Answer(BaseModel):
    answer: str = Field(description="Answer to the question.")

In [10]:
response = client.create(
    messages=[{"role": "user", "content": prompt}],
    response_model=Answer,
)

In [11]:
response

Answer(answer='I am a helpful AI assistant.')

### Native Gemini Structured Outputs

Gemini can also return JSON that follows a Pydantic model schema directly.

In [12]:
native_response = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=prompt,
    config={
        "response_mime_type": "application/json",
        "response_json_schema": Answer.model_json_schema(),
    },
)

In [13]:
native_answer = Answer.model_validate_json(native_response.text)

In [14]:
class AnswerWithReasoning(BaseModel):
    answer: str = Field(description="Answer to the question.")
    reasoning: str = Field(description="Reasoning for the answer.")

In [15]:
response = client.create(
    messages=[{"role": "user", "content": prompt}],
    response_model=AnswerWithReasoning,
)

In [16]:
response

AnswerWithReasoning(answer='I am a helpful assistant.', reasoning="The user asked for my name, and as an AI assistant, I don't have a personal name, so I identify as a helpful assistant.")

### Gemini RAG Pipeline

In [17]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question.")

In [21]:
def get_embedding(text):
    # Use the same local embedding model and Qdrant collection as the project RAG pipeline.
    return embedding_model.encode(text, normalize_embeddings=True).tolist()


def retrieve_data(query, qdrant_client, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings,
    }


def process_context(context):
    return "\n".join(
        f"- ID: {product_id}, rating: {rating}, description: {description}"
        for product_id, description, rating in zip(
            context["retrieved_context_ids"],
            context["retrieved_context"],
            context["retrieved_context_ratings"],
        )
    )


def build_prompt(preprocessed_context, question):
    return f"""You are a shopping assistant that answers questions about the products in stock.
Answer using only the available products below. Refer to them as available products, not as context.

Available products:
{preprocessed_context}

Question:
{question}
"""


def generate_answer(prompt):
    return client.create(
        messages=[{"role": "user", "content": prompt}],
        response_model=RAGGenerationResponse,
    )


def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return {
        "data_object": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"],
    }

In [22]:
output = rag_pipeline("Can you suggest me some earphones?")

In [23]:
output

{'data_object': RAGGenerationResponse(answer='Yes, here are some available products that are earphones:\n\n- **2 Pack iPhone Wired Headphones (ID: B0CH8DRD6K)**: Wired earbuds featuring a 3.5mm jack, stereo sound quality, an ergonomic semi-in-ear design, and a built-in microphone with a remote control.\n- **Volkano Ninja Kids Earphones (ID: B0BBF2VC6X)**: Wired earbuds designed for kids with hearing protection (85dB limit), an inline microphone, cute designs, and an included carrying case with extra earbud tips.\n- **Jesebang Wireless Earbuds (ID: B09X9838WY)**: Wireless Bluetooth 5.2 earbuds with a charging case, touch control, built-in microphones, immersive stereo sound, IP7 waterproof rating, and up to 40 hours of total playtime.'),
 'answer': 'Yes, here are some available products that are earphones:\n\n- **2 Pack iPhone Wired Headphones (ID: B0CH8DRD6K)**: Wired earbuds featuring a 3.5mm jack, stereo sound quality, an ergonomic semi-in-ear design, and a built-in microphone with a

In [24]:
print(output["answer"])

Yes, here are some available products that are earphones:

- **2 Pack iPhone Wired Headphones (ID: B0CH8DRD6K)**: Wired earbuds featuring a 3.5mm jack, stereo sound quality, an ergonomic semi-in-ear design, and a built-in microphone with a remote control.
- **Volkano Ninja Kids Earphones (ID: B0BBF2VC6X)**: Wired earbuds designed for kids with hearing protection (85dB limit), an inline microphone, cute designs, and an included carrying case with extra earbud tips.
- **Jesebang Wireless Earbuds (ID: B09X9838WY)**: Wireless Bluetooth 5.2 earbuds with a charging case, touch control, built-in microphones, immersive stereo sound, IP7 waterproof rating, and up to 40 hours of total playtime.
